# Chapter 3: Classical Language Models

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch03_classical_language_models.ipynb)


## What is in this notebook, and what to change in it

Three cells over one four-sentence corpus, all of it counting. Nothing is
trained and nothing is downloaded.

1. **Bigram counts, and three sentences generated from them.** The probability
   of a word given the one before it is a count divided by a count. The
   generator samples from that conditional distribution rather than taking the
   most likely word, which is why the three sentences it prints differ from each
   other on the same model.
2. **The same model as a full count matrix**, printing P(cat | the) = 0.375, one
   more generated sentence, and a perplexity of 1.8 on a held-out sentence. The
   matrix is mostly zeros, and the zeros are the point: four sentences leave
   most word pairs unseen, and every smoothing method in the chapter exists to
   decide what to put in those cells.
3. **MLE, add-one and Kneser-Ney compared on one held-out sentence**, and this
   is the cell carrying the chapter's argument. The test sentence is "the rug
   sat on the cat", which contains a pair the corpus never shows. MLE assigns
   that pair probability zero and the perplexity prints as **inf**. Add-one
   gives 5.4 and Kneser-Ney 3.5.

An infinite perplexity out of one unseen word pair is the entire reason
smoothing exists, and here it is printed rather than argued.

Change the corpus, which is four lines at the top of each cell, and rerun all
three. Adding a sentence that reuses existing words moves the estimates; adding
one with a new word changes the vocabulary, and with it the size of the table
and every smoothed probability in it. Add the pair "rug sat" to the corpus and
the infinity in cell 3 disappears, which is the cheapest way to see that
smoothing and more data are answers to the same problem.

The instructive edit is subtler. Make one word frequent but always in the same
context, then compare its Kneser-Ney probability against its raw count. The
continuation count Kneser-Ney uses asks how many distinct contexts a word
appears in rather than how often it appears, and a word that is common in only
one place is exactly where the two answers come apart.


> **This notebook was executed when it was built**, so the output under each
> cell is a real run's and you can read the file without running anything.
> Rebuild it with `python tools/build_notebook.py ch03`.


In [1]:
# Seeded before the chapter's own cells run.
#
# The cells below draw on numpy. This notebook is committed with its
# output stored, and a stored number that changes on every rebuild is
# noise printed as a result. The bundle originals are read only and
# cannot be fixed where they live, so they are seeded here instead.
#
# Same seed as tools/claim_instances.py, which produced the slide
# numbers, so a number that appears in both places appears once.
import numpy as np
np.random.seed(20260729)
print('seeded numpy with 20260729')

seeded numpy with 20260729


### 3.1.2 Unigrams, Bigrams, and Beyond

![Figure 3.5 -- Text generation comparison across n-gram orders](../figures/fig-03-5.pdf)


In [2]:
import numpy as np

# Toy corpus with start/end markers
corpus = ["<s> the cat sat on the mat </s>", "<s> the dog sat on the rug </s>",
          "<s> the cat chased the dog </s>", "<s> the dog chased the cat </s>"]
tokens = [s.split() for s in corpus]

def build_bigram(sents):
    counts, ctx = {}, {}
    for s in sents:
        for i in range(1, len(s)):
            counts[(s[i-1], s[i])] = counts.get((s[i-1], s[i]), 0) + 1
            ctx[s[i-1]] = ctx.get(s[i-1], 0) + 1
    return counts, ctx

def generate(counts, ctx, seed="<s>", max_len=12):
    result, w = [], seed
    for _ in range(max_len):
        cands = [(w2, c) for (w1, w2), c in counts.items() if w1 == w]
        if not cands: break
        ws, cs = zip(*cands)
        probs = np.array(cs, dtype=float) / sum(cs)
        w = np.random.choice(ws, p=probs)
        if w == "</s>": break
        result.append(w)
    return " ".join(result)

bi_counts, bi_ctx = build_bigram(tokens)
np.random.seed(42)
for _ in range(3):
    print(f"Generated: {generate(bi_counts, bi_ctx)}")


Generated: the rug
Generated: the cat sat on the dog chased the rug
Generated: the cat sat on the mat


### 3.1.3 Estimating n-gram Probabilities

Here is the complete pipeline (from raw text to generated sentence to perplexity) in code:


In [3]:
import numpy as np

# Toy corpus with start/end tokens
corpus = [
    "<s> the cat sat on the mat </s>",
    "<s> the dog sat on the rug </s>",
    "<s> the cat chased the dog </s>",
    "<s> the dog chased the cat </s>",
]
sents = [s.split() for s in corpus]
vocab = sorted(set(w for s in sents for w in s))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# Build bigram count matrix
C = np.zeros((V, V), dtype=int)
for s in sents:
    for a, b in zip(s, s[1:]):
        C[w2i[a], w2i[b]] += 1
P = C / C.sum(axis=1, keepdims=True).clip(1)  # normalize rows
print("Bigram P(cat|the) =", round(P[w2i["the"], w2i["cat"]], 3))

# Generate a sentence
np.random.seed(7)
w = "<s>"
sent = []
for _ in range(20):
    probs = P[w2i[w]]
    nxt = vocab[np.random.choice(V, p=probs)]
    if nxt == "</s>": break
    sent.append(nxt)
    w = nxt
print("Generated:", " ".join(sent))

# Perplexity on held-out sentence
test = "<s> the cat sat on the rug </s>".split()
log_prob = sum(np.log2(P[w2i[a], w2i[b]]) for a, b in zip(test, test[1:]))
pp = 2 ** (-log_prob / (len(test) - 1))
print(f"Perplexity: {pp:.1f}")


Bigram P(cat|the) = 0.375
Generated: the mat
Perplexity: 1.8


### 3.2.4 Kneser-Ney Smoothing

Here is a code example comparing MLE, add-1, and simplified Kneser-Ney smoothing on our toy corpus:


In [4]:
import numpy as np

corpus = ["<s> the cat sat on the mat </s>", "<s> the dog sat on the rug </s>",
          "<s> the cat chased the dog </s>", "<s> the dog chased the cat </s>"]
sents = [s.split() for s in corpus]
vocab = sorted(set(w for s in sents for w in s))
V = len(vocab)
# Count bigrams and unigrams
bi, uni = {}, {}
for s in sents:
    for a, b in zip(s, s[1:]):
        bi[(a,b)] = bi.get((a,b), 0) + 1; uni[a] = uni.get(a, 0) + 1
# Continuation counts for Kneser-Ney (d=0.75)
pred = {}
for (a, b) in bi: pred.setdefault(b, set()).add(a)
N_types, d = len(bi), 0.75

test = "<s> the rug sat on the cat </s>".split()
for name, method in [("MLE","mle"), ("Add-1","add1"), ("KN","kn")]:
    lp = 0.0
    for a, b in zip(test, test[1:]):
        cab, ca = bi.get((a,b), 0), uni.get(a, 0)
        if method == "mle": p = cab / ca if ca > 0 else 0
        elif method == "add1": p = (cab + 1) / (ca + V)
        else:  # Kneser-Ney
            n_fol = len([w for w in vocab if bi.get((a,w),0)>0])
            p = max(cab-d,0)/ca + d*n_fol/ca * len(pred.get(b,set()))/N_types
        lp += np.log2(p) if p > 0 else float("-inf")
    pp = 2**(-lp/(len(test)-1)) if lp > float("-inf") else float("inf")
    print(f"{name}: perplexity = {pp:.1f}")


MLE: perplexity = inf
Add-1: perplexity = 5.4
KN: perplexity = 3.5


---

## Summary

This notebook demonstrated the key code examples from Chapter 3: Classical Language Models. For the full mathematical exposition and discussion, refer to the textbook chapter.
